# recs_031 — Stage 5 Track B: LLM-judge scores + human calibration (read-only)

Loads artifacts already written by `recs_job_explanation_judge_eval.py` and
`recs_job_explanation_judge_calibration.py --compare`. **No API calls, no re-scoring.**

Prerequisites:
- `python scripts/recs_job_explanation_judge_eval.py configs/recs_job_explanation_judge_eval.json --limit 200` (or your target pool size)
- Hand-label `artifacts/recs/qualitative/user_facing/explanation_judge_calibration_sample.csv`, then `python scripts/recs_job_explanation_judge_calibration.py configs/recs_job_explanation_judge_calibration.json --compare`

See [`docs/plans/rag_extension_plan.md`](../../docs/plans/rag_extension_plan.md) Stage 5 Track B for the full narrative/status, and `recs_030` (`notebooks/ranking/`) for the sibling heuristic-only view this mirrors.

In [1]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd

from steam_review_ml.evaluation.judge_calibration import (
    build_calibration_comparison,
    load_hand_labels,
    summarize_calibration,
)

REPO_ROOT = Path.cwd().parent.parent
RUN_DIR = REPO_ROOT / "artifacts/recs/explanation_eval/runs/latest"
SAMPLE_PATH = REPO_ROOT / "artifacts/recs/qualitative/user_facing/explanation_judge_calibration_sample.csv"

judge_df = pd.read_parquet(RUN_DIR / "explanation_judge_scores.parquet")
heuristic_df = pd.read_parquet(RUN_DIR / "explanation_heuristic_scores.parquet")
print(f"RUN_DIR={RUN_DIR}")
print(f"judge scores: {len(judge_df)} rows, heuristic scores: {len(heuristic_df)} rows")

RUN_DIR=/home/ryanr/workspace/steam_recommendations/artifacts/recs/explanation_eval/runs/latest
judge scores: 200 rows, heuristic scores: 200 rows


## Full-pool judge scores

In [2]:
print(f"judge_faithfulness: mean={judge_df['judge_faithfulness'].mean():.2f} median={judge_df['judge_faithfulness'].median():.1f}")
print(judge_df["judge_faithfulness"].value_counts().sort_index().rename("count"))
print(f"\njudge_relevance: mean={judge_df['judge_relevance'].mean():.2f} median={judge_df['judge_relevance'].median():.1f}")
print(judge_df["judge_relevance"].value_counts().sort_index().rename("count"))

judge_faithfulness: mean=3.16 median=3.0
judge_faithfulness
2    60
3    59
4    70
5    11
Name: count, dtype: int64

judge_relevance: mean=2.49 median=2.0
judge_relevance
1    22
2    89
3    61
4    25
5     3
Name: count, dtype: int64


## Worst-scored explanations (lowest of faithfulness/relevance)

Manual eyeball check -- these are the ones a prompt tweak should target first (mirrors
`recs_030`'s worst-examples cell, using the judge's verdict instead of the cheap heuristic).

In [3]:
worst = judge_df.assign(
    worst_axis=judge_df[["judge_faithfulness", "judge_relevance"]].min(axis=1)
).sort_values("worst_axis").head(10)

for row in worst.to_dict("records"):
    print(
        f"query: {row['query_app_name']} (app_id={row['query_app_id']}) -> rec: {row['rec_app_name']} "
        f"(app_id={row['rec_app_id']}), faithfulness={row['judge_faithfulness']}, relevance={row['judge_relevance']}"
    )
    print(f"  explanation: {row['explanation']}")
    print(f"  rationale: {row['judge_rationale']}")
    print()

query: Among Us (app_id=945360) -> rec: Terraria (app_id=105600), faithfulness=2, relevance=1
  explanation: We think you'll love Terraria because, like Among Us, it's a game that encourages exploration and strategy while also having an element of danger lurking beneath the surface. In Terraria, you'll need to be careful as you dig and build, as you never know what hidden threats or surprises might await you. This sense of uncertainty and potential danger is similar to the suspenseful gameplay found in Among Us, where one player's true intentions are unknown until it's too late.
  rationale: The explanation invents parallels between the games (uncertainty about hidden threats being like not knowing the Impostor) that aren't grounded in the texts, and fails to connect specific mechanics from Among Us (social deduction, voting, multiplayer deception) to Terraria's actual features.

query: Cuphead (app_id=268910) -> rec: The Witcher 3: Wild Hunt (app_id=292030), faithfulness=4, relevance=

## Calibration: judge vs. human labels

Compares the hand-labeled sample (`human_faithfulness`/`human_relevance`) against the judge's
verdicts and the heuristic proxies for the same rows, joined on `example_id` -- the row's
positional index in `explanations.parquet` (see `judge_calibration.build_calibration_comparison`).
Raises if the judge job's `--limit` didn't cover every hand-labeled row.

In [4]:
hand_labels_df = load_hand_labels(SAMPLE_PATH)
comparison_df = build_calibration_comparison(hand_labels_df, heuristic_df, judge_df)
print(f"hand-labeled rows: {len(hand_labels_df)}")

cols = [
    "example_id", "query_app_name", "rec_app_name",
    "human_faithfulness", "judge_faithfulness",
    "human_relevance", "judge_relevance",
]
print(comparison_df[cols].to_string(index=False))

summary = summarize_calibration(comparison_df)
print()
print(json.dumps(summary, indent=2))

hand-labeled rows: 10
 example_id              query_app_name                   rec_app_name  human_faithfulness  judge_faithfulness  human_relevance  judge_relevance
          0                      Kenshi                           Rust                   5                   4                5                3
          1       Insurgency: Sandstorm Tom Clancy's Rainbow Six Siege                   5                   5                5                5
          2                    Terraria                    Garry's Mod                   5                   2                5                2
          3                  The Forest                       Terraria                   5                   3                5                3
          4             Resident Evil 2                    Dying Light                   5                   3                5                4
          5              Stardew Valley                       Terraria                   5                  

## Largest human/judge disagreements

Where the judge and the human labeler disagreed most (sum of absolute gaps on both axes) --
the cases most worth reading closely when deciding how much to trust the judge.

In [5]:
comparison_df = comparison_df.assign(
    disagreement=(comparison_df["human_faithfulness"] - comparison_df["judge_faithfulness"]).abs()
    + (comparison_df["human_relevance"] - comparison_df["judge_relevance"]).abs()
)
worst_disagree = comparison_df.sort_values("disagreement", ascending=False).head(5)

for row in worst_disagree.to_dict("records"):
    print(
        f"example_id={row['example_id']} query: {row['query_app_name']} -> rec: {row['rec_app_name']} | "
        f"human(f={row['human_faithfulness']}, r={row['human_relevance']}) vs "
        f"judge(f={row['judge_faithfulness']}, r={row['judge_relevance']})"
    )
    print(f"  explanation: {row['explanation']}")
    print(f"  judge rationale: {row['judge_rationale']}")
    print()

example_id=2 query: Terraria -> rec: Garry's Mod | human(f=5, r=5) vs judge(f=2, r=2)
  explanation: We think you'll love Garry's Mod because it lets you build and create your own world, just like in Terraria. With its physics sandbox gameplay, you can craft and construct anything from simple objects to complex contraptions, giving you the freedom to express your creativity. Whether you're building a city or creating chaos, Garry's Mod offers endless possibilities for exploration and experimentation.
  judge rationale: The explanation invents details like 'craft' and 'exploration' not present in Garry's Mod's description, and fails to acknowledge that Garry's Mod lacks Terraria's combat, progression, and survival elements that define the user's game.

example_id=6 query: Among Us -> rec: Terraria | human(f=5, r=3) vs judge(f=2, r=1)
  explanation: We think you'll love Terraria because, like Among Us, it's a game that encourages exploration and strategy while also having an element of d